In [4]:
import pandas as pd
import numpy as np

In [5]:
import scipy.stats as st
from scipy.fftpack import fft, fftfreq 
from scipy.signal import argrelextrema
import operator

In [6]:
df=pd.read_csv('database_gas.csv')

In [7]:
df.head()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
0,2019-11-06 11:37:13.038174+01:00,19.48,54.86,128.0,563.0,476.0,731.0,649.0,565.0,128.0
1,2019-11-06 11:37:32.744996+01:00,19.59,54.23,129.0,563.0,477.0,731.0,649.0,565.0,125.0
2,2019-11-06 11:37:53.018087+01:00,19.63,54.05,128.0,566.0,478.0,732.0,649.0,565.0,125.0
3,2019-11-06 11:38:13.093151+01:00,19.64,53.74,128.0,566.0,478.0,732.0,649.0,565.0,125.0
4,2019-11-06 11:38:33.032832+01:00,19.67,53.53,128.0,569.0,480.0,732.0,650.0,565.0,125.0


In [8]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp'] = df['timestamp'].dt.floor(freq='4H')

In [9]:
df.head()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
0,2019-11-06 08:00:00+01:00,19.48,54.86,128.0,563.0,476.0,731.0,649.0,565.0,128.0
1,2019-11-06 08:00:00+01:00,19.59,54.23,129.0,563.0,477.0,731.0,649.0,565.0,125.0
2,2019-11-06 08:00:00+01:00,19.63,54.05,128.0,566.0,478.0,732.0,649.0,565.0,125.0
3,2019-11-06 08:00:00+01:00,19.64,53.74,128.0,566.0,478.0,732.0,649.0,565.0,125.0
4,2019-11-06 08:00:00+01:00,19.67,53.53,128.0,569.0,480.0,732.0,650.0,565.0,125.0


In [10]:
def stat_area_features(x, Te=1.0):

    mean_ts = np.mean(x, axis=1).reshape(-1,1) # mean
    max_ts = np.amax(x, axis=1).reshape(-1,1) # max
    min_ts = np.amin(x, axis=1).reshape(-1,1) # min
    std_ts = np.std(x, axis=1).reshape(-1,1) # std
    skew_ts = st.skew(x, axis=1).reshape(-1,1) # skew
    kurtosis_ts = st.kurtosis(x, axis=1).reshape(-1,1) # kurtosis 
    iqr_ts = st.iqr(x, axis=1).reshape(-1,1) # interquartile rante
    mad_ts = np.median(np.sort(abs(x - np.median(x, axis=1).reshape(-1,1)),
                               axis=1), axis=1).reshape(-1,1) # median absolute deviation
    area_ts = np.trapz(x, axis=1, dx=Te).reshape(-1,1) # area under curve
    sq_area_ts = np.trapz(x ** 2, axis=1, dx=Te).reshape(-1,1) # area under curve ** 2

    return np.concatenate((mean_ts,max_ts,min_ts,std_ts,skew_ts,kurtosis_ts,
                           iqr_ts,mad_ts,area_ts,sq_area_ts), axis=1)

In [11]:
def frequency_domain_features(x, Te=1.0):

    # As the DFT coefficients and their corresponding frequencies are symetrical arrays
    # with respect to the middle of the array we need to know if the number of readings 
    # in x is even or odd to then split the arrays...
    if x.shape[1]%2 == 0:
        N = int(x.shape[1]/2)
    else:
        N = int(x.shape[1]/2) - 1
    xf = np.repeat(fftfreq(x.shape[1],d=Te)[:N].reshape(1,-1), x.shape[0], axis=0) # frequencies
    dft = np.abs(fft(x, axis=1))[:,:N] # DFT coefficients   
    
    # statistical and area features
    dft_features = stat_area_features(dft, Te=1.0)
    # weighted mean frequency
    dft_weighted_mean_f = np.average(xf, axis=1, weights=dft).reshape(-1,1)
    # 5 first DFT coefficients 
    dft_first_coef = dft[:,:5]    
    # 5 first local maxima of DFT coefficients and their corresponding frequencies
    dft_max_coef = np.zeros((x.shape[0],5))
    dft_max_coef_f = np.zeros((x.shape[0],5))
    for row in range(x.shape[0]):
        # finds all local maximas indexes
        extrema_ind = argrelextrema(dft[row,:], np.greater, axis=0) 
        # makes a list of tuples (DFT_i, f_i) of all the local maxima
        # and keeps the 5 biggest...
        extrema_row = sorted([(dft[row,:][j],xf[row,j]) for j in extrema_ind[0]],
                             key=operator.itemgetter(0), reverse=True)[:5] 
        for i, ext in enumerate(extrema_row):
            dft_max_coef[row,i] = ext[0]
            dft_max_coef_f[row,i] = ext[1]    
    
    return np.concatenate((dft_features,dft_weighted_mean_f,dft_first_coef,
                           dft_max_coef,dft_max_coef_f), axis=1)


In [12]:
def convert_to_list(data, column_name):
    Unique_time = data.timestamp.unique()
    X = []
    for time in Unique_time:
        values = data[data.timestamp==time][column_name].tolist()
        if len(values)<721:
            resample_len=721-len(values)
            new_data = np.random.choice(values, resample_len)
            values.extend(new_data)
        X.append(values)
    return np.array(X)

In [13]:
def make_feature(data, column_name):
    # convert data into list of lists
    x=convert_to_list(data, column_name)

    # extract statistical feature
    x_stat=stat_area_features(x)
    x_stat=pd.DataFrame(x_stat)
    x_stat.columns=[column_name+'_stat_'+ str(i) for i in range(x_stat.shape[1])]

    # extract frequency_domain_features
    x_domain=frequency_domain_features(x)
    x_domain=pd.DataFrame(x_domain)
    x_domain.columns=[column_name+'_tm_dom_'+ str(i) for i in range(x_domain.shape[1])]

    # combined both features
    x_final=pd.concat([x_stat, x_domain], axis=1)

    return x_final

In [14]:
# test only for humidity column
humidity=make_feature(df, column_name='humidity')

In [15]:
humidity.head()

,humidity_stat_0,humidity_stat_1,humidity_stat_2,humidity_stat_3,humidity_stat_4,humidity_stat_5,humidity_stat_6,humidity_stat_7,humidity_stat_8,humidity_stat_9,...,humidity_tm_dom_16,humidity_tm_dom_17,humidity_tm_dom_18,humidity_tm_dom_19,humidity_tm_dom_20,humidity_tm_dom_21,humidity_tm_dom_22,humidity_tm_dom_23,humidity_tm_dom_24,humidity_tm_dom_25
0,51.911415,54.86,47.84,2.489106,-0.474746,-1.573329,4.84,0.87,37373.370,1.944416e+06,...,185.111161,155.561125,143.444251,139.367566,136.605620,0.013870,0.178918,0.022191,0.441054,0.262136
1,50.237961,53.03,46.07,1.353464,-0.175723,-0.285546,1.81,0.90,36172.955,1.818655e+06,...,371.184913,191.663604,148.444674,142.313031,95.304279,0.002774,0.006935,0.009709,0.023578,0.015257
2,51.338447,54.72,46.62,1.827435,-0.247580,-0.705200,2.89,1.41,36961.830,1.899872e+06,...,150.437352,145.720155,98.590762,83.512974,60.851192,0.008322,0.012483,0.015257,0.019417,0.031900
3,51.518544,56.37,46.34,3.168485,0.024922,-1.429916,6.02,3.00,37094.215,1.918305e+06,...,643.394460,407.291388,197.314370,169.315219,141.631682,0.004161,0.006935,0.011096,0.019417,0.022191
4,56.479404,57.17,55.44,0.389625,-0.674834,-0.179276,0.46,0.24,40664.965,2.296831e+06,...,61.665862,60.559512,24.629053,24.286357,23.131566,0.004161,0.008322,0.015257,0.026352,0.012483


In [16]:
humidity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 36 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   humidity_stat_0     582 non-null    float64
 1   humidity_stat_1     582 non-null    float64
 2   humidity_stat_2     582 non-null    float64
 3   humidity_stat_3     582 non-null    float64
 4   humidity_stat_4     582 non-null    float64
 5   humidity_stat_5     582 non-null    float64
 6   humidity_stat_6     582 non-null    float64
 7   humidity_stat_7     582 non-null    float64
 8   humidity_stat_8     582 non-null    float64
 9   humidity_stat_9     582 non-null    float64
 10  humidity_tm_dom_0   582 non-null    float64
 11  humidity_tm_dom_1   582 non-null    float64
 12  humidity_tm_dom_2   582 non-null    float64
 13  humidity_tm_dom_3   582 non-null    float64
 14  humidity_tm_dom_4   582 non-null    float64
 15  humidity_tm_dom_5   582 non-null    float64
 16  humidity

In [17]:
humidity.describe()

,humidity_stat_0,humidity_stat_1,humidity_stat_2,humidity_stat_3,humidity_stat_4,humidity_stat_5,humidity_stat_6,humidity_stat_7,humidity_stat_8,humidity_stat_9,...,humidity_tm_dom_16,humidity_tm_dom_17,humidity_tm_dom_18,humidity_tm_dom_19,humidity_tm_dom_20,humidity_tm_dom_21,humidity_tm_dom_22,humidity_tm_dom_23,humidity_tm_dom_24,humidity_tm_dom_25
count,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000,5.820000e+02,...,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000,582.000000
mean,52.248504,54.676203,49.701186,1.214884,-0.113059,-0.076010,1.781168,0.751357,37618.872079,1.979076e+06,...,191.636298,108.934552,77.944316,62.013286,51.036228,0.007245,0.013136,0.017225,0.022179,0.026946
std,4.074089,3.680987,4.799902,0.879481,0.722138,1.880797,1.520801,0.546973,2933.541002,3.060023e+05,...,155.362139,76.692396,51.860385,40.052250,32.085390,0.009250,0.017609,0.007801,0.023048,0.023868
min,42.658946,44.580000,38.160000,0.131168,-4.205225,-1.883091,0.190000,0.100000,30715.470000,1.311129e+06,...,20.990709,14.615055,10.991758,10.162923,7.530259,0.002774,0.002774,0.002774,0.005548,0.002774
25%,48.977205,51.875000,46.070000,0.531779,-0.499882,-0.893062,0.742500,0.360000,35266.048750,1.728401e+06,...,79.423985,51.472580,39.058151,29.962798,24.780477,0.004161,0.008322,0.011442,0.015257,0.019417
50%,52.172607,54.770000,49.540000,1.017423,-0.128853,-0.503002,1.365000,0.590000,37562.252500,1.961350e+06,...,149.066061,89.053985,67.505683,51.950706,43.228419,0.005548,0.011096,0.015257,0.019417,0.024965
75%,55.439879,57.427500,53.842500,1.625310,0.315718,0.190413,2.267500,1.000000,39916.797500,2.213043e+06,...,268.334999,146.383716,105.027270,83.638629,70.262489,0.008322,0.013870,0.020804,0.024965,0.030513
max,62.125423,69.260000,60.120000,5.404560,2.513489,30.412723,11.270000,3.560000,44733.235000,2.781191e+06,...,1387.458860,486.993375,351.049051,349.195079,261.905212,0.205270,0.375867,0.081831,0.441054,0.490985


In [29]:
gdf = pd.DataFrame(df.groupby('timestamp').count()['humidity'])
gdf.reset_index(inplace=True)

In [30]:
gdf.head()

,timestamp,humidity
0,2019-11-06 08:00:00+01:00,69
1,2019-11-06 12:00:00+01:00,720
2,2019-11-06 16:00:00+01:00,719
3,2019-11-06 20:00:00+01:00,720
4,2019-11-07 00:00:00+01:00,720


In [33]:
gdf.rename(columns={"humidity": "count"}, inplace=True)

In [34]:
gdf.head()

,timestamp,count
0,2019-11-06 08:00:00+01:00,69
1,2019-11-06 12:00:00+01:00,720
2,2019-11-06 16:00:00+01:00,719
3,2019-11-06 20:00:00+01:00,720
4,2019-11-07 00:00:00+01:00,720


In [ ]:
# Get for all columns
for column_name in df.columns:
    if column_name != "timestamp":
        fdf=make_feature(df, column_name=column_name)
        gdf=pd.concat([gdf, fdf], axis=1)

In [37]:
gdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 582 entries, 0 to 581
Columns: 326 entries, timestamp to COValue_tm_dom_25
dtypes: datetime64[ns, UTC+01:00](1), float64(324), int64(1)
memory usage: 1.5 MB


In [38]:
gdf.head()

,timestamp,count,temperature_stat_0,temperature_stat_1,temperature_stat_2,temperature_stat_3,temperature_stat_4,temperature_stat_5,temperature_stat_6,temperature_stat_7,...,COValue_tm_dom_16,COValue_tm_dom_17,COValue_tm_dom_18,COValue_tm_dom_19,COValue_tm_dom_20,COValue_tm_dom_21,COValue_tm_dom_22,COValue_tm_dom_23,COValue_tm_dom_24,COValue_tm_dom_25
0,2019-11-06 08:00:00+01:00,69,19.794660,20.20,19.48,0.216603,0.342957,-1.043915,0.32,0.16,...,512.570264,453.814890,441.395203,415.253061,414.560728,0.005548,0.360610,0.094313,0.233010,0.008322
1,2019-11-06 12:00:00+01:00,720,19.937087,20.88,19.00,0.475731,0.051822,-1.003560,0.75,0.39,...,1283.099654,457.815889,419.586974,277.081918,129.401331,0.004161,0.011096,0.006935,0.015257,0.023578
2,2019-11-06 16:00:00+01:00,719,21.988571,24.96,19.11,1.781900,-0.163221,-1.292094,3.64,1.72,...,330.228712,152.252401,151.365369,119.650687,106.600348,0.008322,0.013870,0.019417,0.023578,0.042996
3,2019-11-06 20:00:00+01:00,720,23.152705,25.12,20.92,1.403905,-0.144909,-1.498998,2.78,1.37,...,2614.504362,1053.288613,535.862935,465.742392,376.660996,0.006935,0.013870,0.026352,0.022191,0.038835
4,2019-11-07 00:00:00+01:00,720,20.361761,20.99,19.74,0.323540,0.310539,-0.874969,0.52,0.23,...,663.148084,458.841437,235.416072,231.929754,219.175251,0.009709,0.013870,0.019417,0.022191,0.024965
